# 🌶️ Flask Web Çatısı (Framework) Kapsamlı Python Rehberi

Bu notebook, Python ile web uygulamaları geliştirmek için en çok tercih edilen mikro web çatısı olan **Flask**'i temel kavramlarından ileri düzey oturum yönetimine kadar detaylı şekilde inceleyen bir rehberdir.

> **Not:** Flask, Python standart kütüphanesinde yer almaz. Çalıştırmak için `pip install flask` komutuyla kurulmalıdır.

## 1. Flask Nedir ve Nasıl Çalışır?

Flask, Python ile yazılmış bir **WSGI (Web Server Gateway Interface)** web uygulama çatısıdır. "Mikro" çatı (microframework) olarak adlandırılır çünkü çekirdeğinde veritabanı soyutlama katmanı, form doğrulama veya hazır oturum yönetim araçları gibi katı yapılar barındırmaz. Bunun yerine genişletilebilir bir mimariye sahiptir.

### 1.1. Flask Temel Yapıtaşları

| Bileşen | Açıklama |
|---------|----------|
| **Werkzeug** | HTTP isteklerini, yanıtlarını ve WSGI arayüzünü yöneten güçlü bir kütüphanedir. Routing (yönlendirme) altyapısını sağlar. |
| **Jinja2** | HTML şablonlarını (templates) dinamik olarak işleyen (render) güvenli ve hızlı bir şablon motorudur. |
| **Click** | Flask uygulamaları için komut satırı arayüzü (CLI) araçları oluşturmayı sağlar. |

## 2. Rotalar (Routing) ve Görünüm Fonksiyonları (Views)

Flask'te bir URL'e karşılık gelen Python fonksiyonunu bağlamak için `@app.route` dekoratörü kullanılır.

### 2.1. Temel Rotalar
```python
from flask import Flask

app = Flask(__name__)

@app.route("/")
def index():
    return "Ana Sayfa"

@app.route("/hakkimda")
def about():
    return "Benim Hakkımda"
```

### 2.2. Dinamik Rotalar (URL Değişkenleri)
URL içinden parametre alıp bunu fonksiyona aktarmak mümkündür:
```python
@app.route("/kullanici/<username>")
def show_user(username):
    return f"Kullanıcı Adı: {username}"

@app.route("/post/<int:post_id>")
def show_post(post_id):
    # post_id otomatik olarak tamsayıya (int) dönüştürülür
    return f"Yazı ID: {post_id}"
```

## 3. HTTP Metotları: GET ve POST

Rotalar varsayılan olarak yalnızca `GET` isteklerini karşılar. Kullanıcıdan form verisi almak gibi durumlar için `POST` metoduna da izin verilmelidir.

| Metot | Açıklama | Flask Kullanımı |
|---|---|---|
| **`GET`** | Sunucudan veri okumak/çekmek için kullanılır. Parametreler URL'dedir. | `request.args.get('param')` |
| **`POST`** | Sunucuya veri göndermek/yazmak için kullanılır. Form verileri gövdededir. | `request.form.get('input_name')` |

```python
from flask import request

@app.route("/giris", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        kullanici = request.form.get("kullanici_adi")
        sifre = request.form.get("sifre")
        return f"Hoşgeldin {kullanici}"
    
    # GET ise giriş formunu barındıran HTML render edilir
    return render_template("login.html")
```

## 4. Şablonlar (Templates) ve Jinja2 Motoru

Flask, HTML sayfalarını dinamik verilerle birleştirmek için **Jinja2** şablon motorunu kullanır. HTML dosyaları varsayılan olarak proje dizinindeki `templates/` klasöründe aranır.

### 4.1. Veri Gönderme ve HTML'de Yazdırma
Python:
```python
return render_template("index.html", ad="Alican", liste=["Python", "Flask", "Gemini"])
```

HTML (`index.html`):
```html
<h1>Merhaba {{ ad }}</h1>
<ul>
    {% for eleman in liste %}
        <li>{{ eleman }}</li>
    {% endfor %}
</ul>
```

## 5. Oturum (Session) Yönetimi ve Güvenlik

Flask, kullanıcının sayfalar arası geçişlerinde verilerini korumak için **Session** yapısını sunar. 

- **İstemci Taraflı Çerez (Client-Side Cookie):** Flask session verilerini şifreli bir çereze yazar ve istemci tarayıcısına gönderir. 
- **`secret_key` Gereksinimi:** Çerezlerin istemci tarafından manipüle edilmesini (sahtecilik) engellemek için sunucuda gizli bir anahtar (`secret_key`) tanımlanmalıdır.

```python
from flask import session

app.secret_key = "cok_gizli_bir_anahtar_1234"

# Oturuma veri yazma
session["kullanici"] = "Alican Kaya"

# Oturumdan veri okuma
aktif_kullanici = session.get("kullanici")

# Oturumdan veri silme
session.pop("kullanici", None)
```

> **⚠️ Önemli:** Tarayıcı çerezleri standart olarak en fazla **4KB** boyutunda veri saklayabilir. Bu sebeple session içerisinde çok büyük listeler veya büyük veri yapıları (örneğin devasa sohbet geçmişleri) saklanmamalıdır.

## 6. Yönlendirme (Redirects) ve Hata Yönetimi

Kullanıcıyı başka bir sayfaya yönlendirmek için `redirect` ve dinamik olarak URL oluşturmak için `url_for` kullanılır.

```python
from flask import redirect, url_for

@app.route("/cikis")
def logout():
    session.clear() # Tüm oturumu sıfırla
    # 'chat' isimli rota fonksiyonuna yönlendirir
    return redirect(url_for("chat"))
```

## 7. Uygulamalı Gösterim (Basit Bir Flask Uygulaması Simülasyonu)

Aşağıdaki kod bloğunda, Flask'in Werkzeug test istemcisini kullanarak sunucuyu gerçekten başlatmadan rotalara nasıl sahte (mock) istekler atıp HTTP yanıt kodlarını inceleyebileceğimizi görebilirsiniz.

In [ ]:
from flask import Flask, jsonify, request

# Flask uygulaması tanımlayalım
mock_app = Flask("MockApp")
mock_app.secret_key = "test_secret"

@mock_app.route("/api/bilgi")
def bilgi():
    # Sorgu parametrelerini alma
    param = request.args.get("veri", "varsayilan")
    return jsonify({"durum": "basarili", "alinan_parametre": param})

# Flask'in yerleşik test istemcisini (test client) oluşturuyoruz
with mock_app.test_client() as client:
    print("1. İstek Gönderiliyor: /api/bilgi")
    response = client.get("/api/bilgi")
    print(f"  Durum Kodu : {response.status_code}")
    print(f"  Yanıt      : {response.get_json()}")
    
    print("\n2. İstek Gönderiliyor: /api/bilgi?veri=AlicanKaya")
    response = client.get("/api/bilgi?veri=AlicanKaya")
    print(f"  Durum Kodu : {response.status_code}")
    print(f"  Yanıt      : {response.get_json()}")